In [1]:
import datetime
import json
import urllib.request
import pandas as pd

def main():
    print("서울시 따릉이 데이터")
    
    # 1000개 긁어오기
    init_data = getdataList(1, 1000)
    
    if init_data and "stationInfo" in init_data:
        # list_total_count 활용해서 전체 데이터 긁어오기
        # total_count라서 int걸어주기
        total = int(init_data["stationInfo"].get("list_total_count", 0))
        print(f"전체 대여소 수 : {total}\n")
        
        all_stations = []
        
        #1000개씩 다 긁어오기
        for start_idx in range(1, total + 1, 1000):
            #1000으로하면 에러나서 999
            end_idx = min(start_idx + 999, total)
            print(f"데이터 수집 중... ({start_idx} ~ {end_idx})")
            
            json_data = getdataList(start_idx, end_idx)
            
            if json_data and "stationInfo" in json_data:
                station_data = json_data["stationInfo"].get("row", [])
                #append하면 데이터를 딕셔너리째로 저장해서 안예쁨
                all_stations.extend(station_data)
                
        df = pd.DataFrame(all_stations)
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        csv_filename = f"stationdata_{timestamp}.csv"
        df.to_csv(csv_filename, index=False, encoding='utf-8')
        # json 파일에 저장하기
        json_filename = "seoul_cycle_stations.json"
        with open(json_filename, "w", encoding="utf-8") as f:
            json.dump(all_stations, f, indent=4, ensure_ascii=False)
        
        print(f"총 {len(all_stations)}개의 대여소 정보: {json_filename}")
        
        if all_stations:
            print("\n예시 데이터 두개")
            print(json.dumps(all_stations[:2], indent=4, ensure_ascii=False))
            
    else:
        print("뭔가 오류가 있습니다")

def getRequestUrl(url):
    req = urllib.request.Request(url)
    try:
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            # print("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode("utf-8")
    except Exception as e:
        print(e)
        print("[%s] Error for URL: %s" % (datetime.datetime.now(), url))
        return None
    

def getdataList(start_index = 1, end_index = 1000):
    #서울시는 /로 url 구분해서 형식 맞춰서 썼습니다.
    base_url = "http://openapi.seoul.go.kr:8088"
    api_key = "YOUR_SEOUL_API_KEY"
    req_type = "json"
    service_name = "tbCycleStationInfo"
    url = f"{base_url}/{api_key}/{req_type}/{service_name}/{start_index}/{end_index}/"
    # print(url)

    responseDecode = getRequestUrl(url)

    if responseDecode is None:
        return None
    else:
        try:
            return json.loads(responseDecode)
        except json.JSONDecodeError:
            print(f"[{start_index}~{end_index}] JSON 파싱 에러 발생")
            return None

if __name__ == '__main__':
    main()


서울시 따릉이 데이터
전체 대여소 수 : 3230

데이터 수집 중... (1 ~ 1000)
데이터 수집 중... (1001 ~ 2000)
데이터 수집 중... (2001 ~ 3000)
데이터 수집 중... (3001 ~ 3230)
총 3230개의 대여소 정보: seoul_cycle_stations.json

예시 데이터 두개
[
    {
        "STA_LOC": "마포구",
        "RENT_ID": "ST-10",
        "RENT_NO": "00108",
        "RENT_NM": "서교동 사거리",
        "RENT_ID_NM": "108. 서교동 사거리",
        "HOLD_NUM": "12",
        "STA_ADD1": "서울특별시 마포구 양화로 93",
        "STA_ADD2": "427",
        "STA_LAT": "37.55274582",
        "STA_LONG": "126.91861725",
        "START_INDEX": 0,
        "END_INDEX": 0,
        "RNUM": "1"
    },
    {
        "STA_LOC": "광진구",
        "RENT_ID": "ST-100",
        "RENT_NO": "00503",
        "RENT_NM": "더샵스타시티 C동 앞",
        "RENT_ID_NM": "503. 더샵스타시티 C동 앞",
        "HOLD_NUM": "15",
        "STA_ADD1": "서울특별시 광진구 아차산로 262",
        "STA_ADD2": "더샵스타시티 C동 앞",
        "STA_LAT": "37.53666687",
        "STA_LONG": "127.07359314",
        "START_INDEX": 0,
        "END_INDEX": 0,
        "RNUM": "2"
    }
]
